# LoRA fine-tuning of an out-of-catalog VLM - end to end

This notebook runs the whole chain. It demonstrates six bricks and **nothing else**:

| # | Brick |
|---|---|
| 1 | A model **absent from the Azure AI catalog** becomes a governed AML asset, pinned to a SHA |
| 2 | A **versioned dataset**, built from time-series telemetry |
| 3 | A **LoRA pipeline** on spot GPU |
| 4 | **Two artifacts** from one run: the adapter and the merged weights, side by side with their size |
| 5 | An **AML endpoint** that serves it |
| 6 | The adapter **detached**, running outside Azure |

> Deliberately out of scope: fine-tuning quality, scalability, endpoint robustness, monitoring,
> blue/green. Success here means "the chain runs end to end and every link is visible",
> not "the model is good".

## Before you start

Three things, once. Full details in [../../README.md](../../README.md).

1. **Python 3.10 - 3.12 kernel.** Not 3.13/3.14: `torch` and `peft` have no wheels for them yet.
   ```
   python -m pip install -r ../../requirements.txt -r ../../requirements-local.txt
   ```
2. **Copy `.env.example` to `.env`** at the repo root and fill it in. That is the only file you edit.
3. **`az login --tenant <your-tenant-id>`** in a terminal.

The only real prerequisite is **an Azure subscription you can create resources in**. The resource
group, the workspace, both compute clusters, the dataset, the environments and the endpoint are all
created by the cells below - name them in `.env` and they get provisioned if they do not exist.
Set `CREATE_IF_MISSING = False` in the next cell if you would rather it failed than provisioned.

The same cell also absorbs what a policy-managed tenant does to a fresh workspace: it forces
identity-based datastore auth (shared keys are commonly disabled), and if the workspace storage
comes out unreachable it opens it back up - firewall closed to this machine's IP first, public
access second, **allow-list dropped third**. That last step looks like a regression and is not:
AML compute nodes are not on your IP and are not covered by `bypass: AzureServices`, so an
allow-list that keeps you in keeps every job out. `OPEN_STORAGE_TO_MY_IP = False` turns the whole
repair off, which is what you want when you run from inside the VNet.

Cells marked **$** switch on billable resources. Cells marked **teardown** switch them off.


In [ ]:
# Run once, on the 3.10-3.12 kernel. Skip it if you already installed the two
# requirement files into your venv.
#   requirements.txt       -> orchestration + local dataset build (bricks 1 to 5)
#   requirements-local.txt -> torch CPU + transformers + peft (brick 6)
#
# Absolute paths, resolved from wherever the kernel happens to start: VS Code and
# Jupyter disagree about the working directory of a notebook, and a relative
# "-r ../../requirements.txt" silently works for one of them only.
import sys
from pathlib import Path

_repo = Path.cwd()
while not (_repo / "requirements.txt").is_file() and _repo != _repo.parent:
    _repo = _repo.parent

!{sys.executable} -m pip install -q -r "{_repo}/requirements.txt" -r "{_repo}/requirements-local.txt"


In [ ]:
import json
import os
import shutil
import subprocess
import urllib.request
from pathlib import Path

from azure.ai.ml import MLClient
from azure.ai.ml.entities import Workspace
from azure.core.exceptions import ResourceNotFoundError
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

REPO = Path.cwd()
while not (REPO / "finetune").is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
assert (REPO / "finetune").is_dir(), f"Repo root not found from {Path.cwd()}"

# ONE file to fill in: .env at the repo root, copied from .env.example.
# It is gitignored, so workspace identifiers never travel with the code.
# Real environment variables win over the file (override=False), which is what a
# CI run wants.
load_dotenv(REPO / ".env", override=False)

REQUIRED = ["AZURE_SUBSCRIPTION_ID", "AZURE_RESOURCE_GROUP", "AZUREML_WORKSPACE_NAME"]
missing = [k for k in REQUIRED if not os.environ.get(k)]
if missing:
    raise RuntimeError(
        f"Missing {', '.join(missing)}.\n"
        f"Copy {REPO / '.env.example'} to {REPO / '.env'} and fill in the values.\n"
        "See README.md > Quickstart."
    )

SUBSCRIPTION = os.environ["AZURE_SUBSCRIPTION_ID"]
RESOURCE_GROUP = os.environ["AZURE_RESOURCE_GROUP"]
WORKSPACE = os.environ["AZUREML_WORKSPACE_NAME"]
LOCATION = os.environ.get("AZURE_LOCATION", "")

# Set to False to fail fast instead of provisioning anything.
CREATE_IF_MISSING = True

# On a policy-managed tenant the workspace storage is created with public network
# access DISABLED, and every upload from a laptop then fails. This repairs it by
# opening the account to THIS MACHINE'S IP ONLY. Set to False to be told about the
# problem instead - which is what you want when you run from inside the VNet, where
# it is not a problem at all.
OPEN_STORAGE_TO_MY_IP = True

# Everything below reads from here. Cluster names are NOT configurable: the
# pipeline YAML references them by name, so a rename would have to happen in two
# places and would break the demo in a way that only shows up at submission.
ENDPOINT = os.environ.get("ENDPOINT_NAME", "maint-vlm-ep")
MAX_SAMPLES = int(os.environ.get("MAX_SAMPLES", "0"))
GPU_CLUSTER = "gpu-cluster-spot"
CPU_CLUSTER = "cpu-cluster"
DATASET_NAME = "maintenance_vlm_ds"
DATASET_VERSION = "1"

credential = DefaultAzureCredential()
ml_client = MLClient(credential, SUBSCRIPTION, RESOURCE_GROUP, WORKSPACE)

# Resolved once, at cell scope: it is needed on the creation path below AND by the
# storage reachability check at the end, which runs whether or not we created
# anything.
az = shutil.which("az")

try:
    workspace = ml_client.workspaces.get(WORKSPACE)
    print(f"workspace : {workspace.name} ({workspace.location})  exists")
except ResourceNotFoundError:
    # Before believing "it does not exist", check that we are even looking at the
    # right subscription. A drifted `az` session reports a perfectly real resource
    # group as missing, and the reflex is to doubt Azure rather than the token.
    # Without this guard, that drift would end in a brand new workspace in the
    # wrong subscription - an expensive way to learn the lesson.
    if az is None:
        # Not optional, and not silently skippable: `az` is both the drift guard
        # below and the only thing here that can create a resource group - the AML
        # SDK creates workspaces, never their container. Skipping it would let the
        # cell fail later with a "resource group not found" that points nowhere.
        raise RuntimeError(
            "The Azure CLI was not found on PATH, and it is needed to verify the active "
            "subscription and to create the resource group.\n"
            "Install it, then run: az login --tenant <your-tenant-id>"
        )

    active = json.loads(subprocess.run([az, "account", "show", "-o", "json"],
                                       capture_output=True, text=True, check=True).stdout)
    if active["id"].lower() != SUBSCRIPTION.lower():
        raise RuntimeError(
            f"az is logged into subscription {active['id']} ({active['name']}), "
            f"but .env says {SUBSCRIPTION}.\n"
            f"Run: az account set --subscription {SUBSCRIPTION}"
        )

    if not CREATE_IF_MISSING:
        raise
    if not LOCATION:
        raise RuntimeError(
            f"Workspace '{WORKSPACE}' does not exist and AZURE_LOCATION is not set.\n"
            "Add AZURE_LOCATION to .env (e.g. westeurope) to let this cell create it."
        )

    print(f"workspace : {WORKSPACE} not found -> creating it in {LOCATION} (~3 min)")
    # The SDK creates the workspace, not the resource group that must contain it.
    # `az group create` is idempotent, so this is also the path when the group
    # already exists but the workspace does not.
    subprocess.run([az, "group", "create", "-n", RESOURCE_GROUP, "-l", LOCATION, "-o", "none"], check=True)
    print(f"group     : {RESOURCE_GROUP} ready in {LOCATION}")
    workspace = ml_client.workspaces.begin_create(
        Workspace(
            name=WORKSPACE,
            location=LOCATION,
            description="LoRA fine-tuning of an out-of-catalog VLM - demo workspace.",
            # NOT the SDK default, which is "accesskey". Corporate policy routinely
            # disables shared-key access on storage accounts; a workspace left in
            # accesskey mode then hands out a SAS derived from a key the account
            # refuses. Nothing complains here - it surfaces three cells later, on
            # the dataset upload, as `KeyBasedAuthenticationNotPermitted`.
            system_datastores_auth_mode="identity",
        )
    ).result()
    print(f"workspace : {workspace.name} ({workspace.location})  created")

# The same trap on a workspace that already existed - including one created by an
# earlier run of this notebook, before the line above existed. The switch is safe
# in both directions: identity-based access works whether or not shared keys are
# allowed, and it is what the cluster role assignments in the next cell are for.
auth_mode = (getattr(workspace, "system_datastores_auth_mode", "") or "").lower()
if auth_mode != "identity":
    print(f"workspace : datastore auth is '{auth_mode or 'unset'}' -> switching to identity")
    workspace = ml_client.workspaces.begin_update(
        workspace, system_datastores_auth_mode="identity"
    ).result()
print(f"workspace : datastore auth = {workspace.system_datastores_auth_mode}")

# Is the workspace storage even reachable from here? Checked now rather than
# discovered 100 MB into the first upload. Azure Policy on many corporate tenants
# creates storage accounts with public network access disabled, and every asset
# upload from a laptop then dies on a 403 `AuthorizationFailure` - a code that
# reads like a permissions problem (the RBAC one is `AuthorizationPermissionMismatch`)
# and sends you re-granting roles you already hold.
STORAGE = workspace.storage_account.split("/")[-1]


def storage_net() -> dict:
    """publicNetworkAccess + firewall default, as the service reports them right now."""
    out = subprocess.run(
        [az, "storage", "account", "show", "-n", STORAGE, "-g", RESOURCE_GROUP,
         "--query", "{public:publicNetworkAccess,default:networkRuleSet.defaultAction}", "-o", "json"],
        capture_output=True, text=True).stdout
    return json.loads(out or "{}")


def exempt_resource_group_from_modify_policy() -> bool:
    """Waive the `modify` policy that keeps re-disabling public access, for THIS
    resource group only.

    This is the step that removes the need for a VNet. A `modify` policy is
    usually assigned at a management group, which you cannot touch - but an
    exemption can be created at any scope BELOW the assignment, and Owner of the
    resource group is enough. So the demo waives the rule on its own throwaway
    group and leaves the rest of the tenant exactly as it was.
    """
    acct = subprocess.run([az, "storage", "account", "show", "-n", STORAGE, "-g", RESOURCE_GROUP,
                           "--query", "id", "-o", "tsv"], capture_output=True, text=True).stdout.strip()
    states = json.loads(subprocess.run(
        [az, "policy", "state", "list", "--resource", acct, "--query",
         "[?policyDefinitionAction=='modify'].{assignment:policyAssignmentId,"
         "ref:policyDefinitionReferenceId,name:policyDefinitionName}", "-o", "json"],
        capture_output=True, text=True).stdout or "[]")
    # Several modify policies usually apply (local auth, anonymous blob access...).
    # Waive ONLY the one about public network access - the others are none of this
    # demo's business and disabling them would be gratuitous.
    hit = next((s for s in states if "publicnetwork" in (s.get("name") or "").lower()), None)
    if not hit:
        return False

    print(f"            a modify policy is rewriting it: {hit['name']}")
    print(f"            waiving it for resource group {RESOURCE_GROUP} only")
    cmd = [az, "policy", "exemption", "create", "--name", "exempt-storage-public-network",
           "--resource-group", RESOURCE_GROUP, "--policy-assignment", hit["assignment"],
           "--exemption-category", "Waiver",
           "--description", "AML demo: IP-restricted laptop access to the workspace storage.",
           "-o", "none"]
    if hit.get("ref"):
        cmd += ["--policy-definition-reference-ids", hit["ref"]]
    done = subprocess.run(cmd, capture_output=True, text=True)
    if done.returncode:
        tail = done.stderr.strip().splitlines()[-1] if done.stderr.strip() else "no stderr"
        print(f"            exemption refused - {tail}")
        return False
    return True


def open_public_access() -> bool:
    """Enable public access and VERIFY by reading the account back.

    Never trust the exit code here. A policy with a `modify` effect does not
    reject the write - it rewrites the property inside the request, so the PUT
    returns 200 on a value the service never stored. Believing the return code
    would make this cell announce success and send you into brick 1 against a
    storage account that has not moved.
    """
    subprocess.run([az, "storage", "account", "update", "-n", STORAGE, "-g", RESOURCE_GROUP,
                    "--public-network-access", "Enabled", "-o", "none"],
                   capture_output=True, text=True)
    return storage_net().get("public") == "Enabled"


def allow_by_default() -> bool:
    """Drop the IP allow-list once public access is on, and VERIFY.

    Counter-intuitive, and it cost two dead pipelines: an IP allow-list locks out
    AML's own compute. Cluster nodes are not in your VNet and they are NOT covered
    by `bypass: AzureServices` - that bypass serves control-plane services, not the
    VMs running your jobs. With `defaultAction: Deny` the node can neither read the
    build context nor upload a single log line, so the job dies after ~8 minutes
    leaving an EMPTY log, which names nothing and points nowhere.

    `Allow` is also how these accounts ship. It exposes the endpoint, not the data:
    the data plane still demands an Entra token, because tenant policy has already
    disabled shared-key auth and anonymous blob access.
    """
    subprocess.run([az, "storage", "account", "update", "-n", STORAGE, "-g", RESOURCE_GROUP,
                    "--default-action", "Allow", "-o", "none"],
                   capture_output=True, text=True)
    return storage_net().get("default") == "Allow"


net = storage_net() if az else {}

if net.get("public") == "Disabled" and not OPEN_STORAGE_TO_MY_IP:
    print(f"storage   : {STORAGE} - PUBLIC NETWORK ACCESS DISABLED (OPEN_STORAGE_TO_MY_IP is False)")
    print("            Uploads from this machine will 403 with AuthorizationFailure.")
    print("            Run from inside the VNet, or flip the flag - see README > Troubleshooting.")
elif net.get("public") == "Disabled":
    # The ORDER below is the entire safety argument. These accounts ship with
    # networkRuleSet.defaultAction = "Allow", so enabling public access FIRST
    # would expose the account to the whole internet for as long as the next
    # command takes to run. So: close the firewall and add the IP rule, THEN open
    # public access - onto a door already shut to everyone but this machine.
    # Never run the --public-network-access line on its own.
    my_ip = os.environ.get("STORAGE_ALLOW_IP", "").strip()
    if not my_ip:
        try:
            my_ip = urllib.request.urlopen("https://api.ipify.org", timeout=10).read().decode().strip()
        except Exception as exc:  # corporate proxies routinely block this
            print(f"storage   : cannot determine this machine's egress IP ({exc.__class__.__name__})")
            print("            Put it in STORAGE_ALLOW_IP in .env and re-run this cell.")

    if my_ip:
        print(f"storage   : {STORAGE} - public access disabled -> opening it to {my_ip} only")
        subprocess.run([az, "storage", "account", "update", "-n", STORAGE, "-g", RESOURCE_GROUP,
                        "--default-action", "Deny", "--bypass", "AzureServices", "-o", "none"], check=True)
        subprocess.run([az, "storage", "account", "network-rule", "add", "--account-name", STORAGE,
                        "-g", RESOURCE_GROUP, "--ip-address", my_ip, "-o", "none"], check=True)

        # Second attempt only if the first silently did nothing, and only after
        # removing the cause. Retrying the same command without the exemption
        # would loop forever on a 200 that changes nothing.
        opened = open_public_access() or (exempt_resource_group_from_modify_policy() and open_public_access())

        if opened:
            # Public access is on. NOW drop the allow-list: it was scaffolding for
            # the window above, and leaving it in place would block every AML job.
            widened = allow_by_default()
            state = "open (firewall default: Allow)" if widened else f"reachable from {my_ip} only"
            print(f"storage   : {STORAGE} - {state}")
            if not widened:
                print("            WARNING: could not drop the IP allow-list. AML compute nodes")
                print("            are not on this IP, so every job will fail with an empty log.")
            print("            Firewall rules take up to a minute to propagate.")
            print(f"            revert: az storage account update -n {STORAGE} -g {RESOURCE_GROUP} "
                  "--public-network-access Disabled")
            print(f"                    az policy exemption delete --name exempt-storage-public-network "
                  f"-g {RESOURCE_GROUP}")
        else:
            print(f"storage   : {STORAGE} - STILL DISABLED, and the exemption did not go through.")
            print("            Nothing more can be done from this machine: the account is")
            print("            private-endpoint-only. Run the notebook on an AML compute instance")
            print("            with the workspace managed VNet on - see README > Troubleshooting.")
elif net.get("default") == "Deny":
    # Reached on a re-run, or on an account somebody firewalled by hand. Same
    # trap: the allow-list keeps YOU in and keeps AML's compute out.
    print(f"storage   : {STORAGE} - firewall on, allow-listed IPs only -> AML compute is locked out")
    if OPEN_STORAGE_TO_MY_IP and allow_by_default():
        print(f"storage   : {STORAGE} - firewall default set to Allow, jobs can reach it again")
    else:
        print("            Leave it and every job dies after ~8 min with an empty log.")
elif net:
    print(f"storage   : {STORAGE} - reachable")

print(f"repo      : {REPO}")
print(f"endpoint  : {ENDPOINT}   max_samples: {MAX_SAMPLES or 'all'}")


---
## Compute: one GPU cluster, one CPU cluster

Created here if they do not exist. Both scale to **zero** and stay there until a job asks for a
node, so nothing below bills at rest.

Three details are the difference between a pipeline that runs and one that dies at minute zero:

- **`tier="low_priority"` on the GPU.** Dedicated GPU quota is commonly **0** on modern families,
  and that is not a blocker: low-priority is a *separate quota pool*. Price of admission is
  preemption - acceptable for 10 minutes of training, which is why the endpoint later runs on CPU.
- **A system-assigned identity on the cluster, with `Storage Blob Data Contributor` on the
  workspace storage.** When the workspace uses identity-based datastore access, a cluster without
  it dies in ~20 s with `UserError: Identity of the specified managed compute ... is not found` -
  an error that says nothing about storage and sends you looking in the wrong place.
- **Quota is enforced at *allocation*, not at creation.** A cluster is created happily against a
  quota of 0, and you find out 30 minutes later. So the cell prints the quota instead of letting
  you assume it: check the low-priority row before moving on.


In [ ]:
from azure.ai.ml.entities import AmlCompute, IdentityConfiguration


def ensure_cluster(name: str, size: str, tier: str) -> AmlCompute:
    """Create the cluster if it is missing, and report it either way. Re-runnable."""
    try:
        cluster = ml_client.compute.get(name)
        print(f"{name:18s} exists   size={cluster.size} tier={cluster.tier} min={cluster.min_instances}")
        return cluster
    except ResourceNotFoundError:
        pass

    cluster = ml_client.compute.begin_create_or_update(
        AmlCompute(
            name=name,
            type="amlcompute",
            size=size,
            tier=tier,
            min_instances=0,
            max_instances=1,
            # The only real cost guarantee: the node is released with no human in
            # the loop. Idle time bills at the full node rate, so shorter is
            # cheaper - do not raise this to 1800 "for safety", it protects nothing.
            idle_time_before_scale_down=300,
            # snake_case on purpose: the SDK pascal-cases this string itself, so
            # "SystemAssigned" would be sent as the invalid "Systemassigned".
            identity=IdentityConfiguration(type="system_assigned"),
        )
    ).result()
    print(f"{name:18s} created  size={cluster.size} tier={cluster.tier} min=0 idle=300s")
    return cluster


gpu = ensure_cluster(GPU_CLUSTER, "Standard_NC4as_T4_v3", "low_priority")
cpu = ensure_cluster(CPU_CLUSTER, "Standard_DS3_v2", "dedicated")

# The moment the workspace storage sits behind a firewall, AML stops building
# environment images with ACR Tasks - the build agent has no route to the build
# context - and switches to "image build on compute". If no compute is nominated
# for that, the build fails before a single layer is pulled, and brick 3 dies on a
# JobException whose only artifact is a three-line 20_image_build_log.txt pointing
# at an image build run that has no logs of its own. Nothing anywhere says
# "storage firewall". The CPU cluster is the right host: it already holds Storage
# Blob Data Contributor on that account, granted below.
if storage_net().get("default") == "Deny":
    ws = ml_client.workspaces.get(WORKSPACE)
    if ws.image_build_compute != CPU_CLUSTER:
        print(f"\nimage builds     : storage is firewalled -> building images on {CPU_CLUSTER}")
        ml_client.workspaces.begin_update(ws, image_build_compute=CPU_CLUSTER).result()
    else:
        print(f"\nimage builds     : on {CPU_CLUSTER} (required while the storage is firewalled)")

# Quota, reported here rather than discovered at submission. Creating a cluster
# succeeds even with a quota of 0: the limit is enforced when a node is ALLOCATED,
# so a green cell above proves nothing on its own. What matters for this demo is
# the low-priority pool, which is counted separately from the dedicated one - a
# dedicated NC quota of 0 is normal and not a blocker.
print("\nquota (low-priority is the pool this demo uses):")
for usage in ml_client.compute.list_usage():
    # Three traps in this payload, and every one of them misleads:
    #   name          -> stays a raw {"value": ..., "localizedValue": ...} dict instead
    #                    of a UsageName ('dict' object has no attribute 'value')
    #   current_value -> stays None, while the real number sits beside it under its
    #                    camelCase REST name `currentValue` (int(None) raises)
    #   name.value    -> IDENTICAL for the dedicated and the low-priority row. Only
    #                    `usage.type` tells them apart, so filtering on the label alone
    #                    prints the same line twice and flags a dedicated quota of 0 as
    #                    a problem - the exact false alarm this cell exists to prevent.
    pool = ("low-priority" if "lowPriorityCores" in usage.type
            else "dedicated" if "dedicatedCores" in usage.type else "")
    if not pool:
        continue
    raw = usage.name
    label = (raw.get("value") if isinstance(raw, dict) else getattr(raw, "value", None)) or ""
    if "NCAS" not in label:
        continue
    used = usage.current_value if usage.current_value is not None else getattr(usage, "currentValue", 0)
    # -1 means "no limit", not "minus one core".
    limit = "unlimited" if usage.limit == -1 else str(int(usage.limit or 0))
    flag = ""
    if usage.limit == 0:
        flag = ("  <-- request an increase before running the pipeline" if pool == "low-priority"
                else "  (normal, and NOT a blocker - this demo runs low-priority)")
    print(f"  {label:28s} {pool:13s} {int(used or 0):>3} / {limit:>9} cores{flag}")
print("  increase: Studio > Quota, or https://aka.ms/azureml-quota-request")

# Data-plane RBAC on the workspace storage. Needs Owner or User Access
# Administrator on the storage account; if you do not have it, the failures below
# are printed rather than raised so the cell is not a dead end - but with
# identity-based datastores they ARE fatal further down, so read the output.
storage_id = ml_client.workspaces.get(WORKSPACE).storage_account
az = shutil.which("az")
if not az:
    print("\naz CLI not on PATH - skipping the role assignments. See README > Troubleshooting.")
else:
    print()


def grant_blob_contributor(label: str, principal: str, principal_type: str) -> None:
    """Idempotent. Prints the failure rather than raising - see the note above."""
    done = subprocess.run(
        [az, "role", "assignment", "create",
         "--assignee-object-id", principal,
         "--assignee-principal-type", principal_type,
         "--role", "Storage Blob Data Contributor",
         "--scope", storage_id, "-o", "none"],
        capture_output=True, text=True,
    )
    if done.returncode == 0:
        print(f"{label:18s} granted Storage Blob Data Contributor on the workspace storage")
    elif "RoleAssignmentExists" in done.stderr:
        print(f"{label:18s} role already present")
    else:
        print(f"{label:18s} COULD NOT grant the role - see README > Troubleshooting")
        print(f"  principal={principal}")
        print(f"  {done.stderr.strip().splitlines()[-1] if done.stderr.strip() else 'no stderr'}")


for cluster in (gpu, cpu) if az else []:
    principal = getattr(cluster.identity, "principal_id", None)
    if not principal:
        print(f"{cluster.name:18s} no managed identity - re-run this cell if a job fails on identity")
        continue
    grant_blob_contributor(cluster.name, principal, "ServicePrincipal")

# And you need it too, not only the clusters. With identity-based datastores the
# dataset upload in brick 2 travels on YOUR token instead of on an account key, so
# without this it fails with a flat 403 that names no missing role. Data-plane role
# assignments also take a minute or two to propagate: a 403 on the very next cell
# usually just means you were faster than Entra ID.
if az:
    me = subprocess.run([az, "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"],
                        capture_output=True, text=True)
    if me.returncode == 0 and me.stdout.strip():
        grant_blob_contributor("you", me.stdout.strip(), "User")
    else:
        print(f"{'you':18s} could not resolve your own object id (signed in as a service principal?)")
        print("                   grant yourself Storage Blob Data Contributor manually if the upload 403s")


---
## Brick 1: bring a model that is not in the catalog into AML

`SmolVLM-256M-Instruct` does not exist in the Azure AI catalog. We download it **once**, pinned to its
**commit SHA**, and register it as a `custom_model` asset.

Two details carry everything that follows:

- **Never call `from_pretrained("org/repo")` inside the training job.** AML compute has no guaranteed
  internet access; that single line is the number one reason a fine-tuning job dies at minute zero.
  We mount the asset instead.
- **Pin by SHA, never by `main`.** Otherwise the asset registered on Monday and the one retrained from
  on Friday can be different weights under the same name, with nothing in the registry saying so.


In [ ]:
import sys

sys.path.insert(0, str(REPO / "finetune" / "models"))
from fetch_base_model import DEFAULT_REPO, fetch  # noqa: E402

BASE_DIR = REPO / "tmp-base-model"
source = fetch(DEFAULT_REPO, str(BASE_DIR))
source

In [ ]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Model

base_model = ml_client.models.create_or_update(
    Model(
        name="base-smolvlm",
        path=str(BASE_DIR),
        type=AssetTypes.CUSTOM_MODEL,
        description="SmolVLM-256M-Instruct mirrored from the Hugging Face Hub. Not in the Azure AI catalog.",
        tags={
            "source": "huggingface",
            "repo": source["repo"],
            "revision": source["revision"],
            "license": "apache-2.0",
            "in_azure_catalog": "false",
        },
    )
)
print(f"{base_model.name}:{base_model.version}  {source['size_mb']} MB  revision={source['revision'][:12]}")
print("-> show this in the Studio model registry")


---
## Brick 2: the versioned dataset

620 chart images - 4 sensors over 30 minutes - each paired with the JSON work order it should
produce. **Chronological** 500 / 60 / 60 split: a random split would leak train into test, because
the windows overlap (5-minute stride for a 30-minute window), and the evaluation would be worthless.

The input is committed: [`finetune/data/telemetry_sample.csv`](../data/telemetry_sample.csv), 24
hours of 4 machines, six columns you can open and edit to watch a label change. The **images are
not committed**: windowing, labelling and index assignment are deterministic, so the PNGs are a pure
function of the CSV and are rebuilt here in under a minute.

The cell below does both halves, and neither needs anything outside this repo:

1. builds the dataset locally (free, no Azure),
2. uploads it to `workspaceblobstore` and registers it as a versioned `uri_folder` asset.

> Data asset **versions are immutable**. Re-running finds the existing `v1` and leaves it alone.
> To publish a changed dataset, bump `DATASET_VERSION`.


In [ ]:
from azure.ai.ml.entities import Data

# Inside the repo, not %TEMP%: brick 6 scores the test split from this same folder,
# and a demo that depends on the temp directory surviving is a demo that breaks on
# the second day. tmp-*/ is gitignored.
DATASET_LOCAL = REPO / "tmp-dataset"

if (DATASET_LOCAL / "manifest.json").is_file():
    print(f"dataset already built: {DATASET_LOCAL}")
else:
    subprocess.run(
        [sys.executable, str(REPO / "finetune" / "data" / "build_dataset.py"),
         "--telemetry", str(REPO / "finetune" / "data" / "telemetry_sample.csv"),
         "--out", str(DATASET_LOCAL)],
        check=True,
    )

n_images = len(list((DATASET_LOCAL / "images").glob("*.png")))
print(f"{n_images} charts in {DATASET_LOCAL}")

try:
    dataset = ml_client.data.get(DATASET_NAME, version=DATASET_VERSION)
    print("already registered (versions are immutable - bump DATASET_VERSION to republish)")
except ResourceNotFoundError:
    dataset = ml_client.data.create_or_update(
        Data(
            name=DATASET_NAME,
            version=DATASET_VERSION,
            type=AssetTypes.URI_FOLDER,
            path=str(DATASET_LOCAL),  # a local path here means "upload it for me"
            description="Chart images + JSONL work orders rendered from 30-min, 4-sensor telemetry windows.",
            tags={"source": "finetune/data/telemetry_sample.csv", "split": "chronological 500/60/60"},
        )
    )
    print("registered")

print(f"{dataset.name}:{dataset.version}  {dataset.type}")
print(dataset.path)


---
## Brick 3: the LoRA pipeline

Two environments, built **on top of the curated images** rather than from a conda file: a conda file
would create a new environment inside the image and silently shadow the torch/CUDA build that the
ACPT image exists precisely to provide.

| environment | base image | used by |
|---|---|---|
| `vlm-lora-env` | `acpt-pytorch-2.2-cuda12.1:57` | `train_lora`, on the T4 |
| `vlm-serve-env` | `minimal-py311-inference:60` | `merge_lora` and the endpoint, on CPU |

Both Dockerfiles pin **`numpy<2`**, and that pin is load-bearing. Nothing in `transformers`/`peft`
requires numpy 2, but nothing forbids it either, so pip upgrades the image's numpy 1.x in place - and
the image's `scipy` still does `from numpy import Inf`, an alias deleted in numpy 2.0. A LoRA job
then dies inside an object-detection loss helper, 25 minutes and one GPU node later.
**When you layer pip onto a curated image, pin what the image already depends on.**

Environments are **immutable**: re-registering the same version with a changed Dockerfile keeps
serving the old image, and the fix looks like it did nothing. So the YAMLs carry no `version` (AML
auto-increments) and the components reference `@latest`.

The first build takes ~10 min. Later ones reuse the Docker layer cache.


In [ ]:
from azure.ai.ml import load_environment

# Each run creates a NEW version (the YAMLs declare no version on purpose).
# That is the point: it is the only way a Dockerfile fix actually reaches the job.
envs = {}
for filename in ("vlm-lora-env.yml", "vlm-serve-env.yml"):
    created = ml_client.environments.create_or_update(
        load_environment(REPO / "finetune" / "environments" / filename)
    )
    envs[created.name] = created.version
    print(f"{created.name}:{created.version}")


### $ Submitting the pipeline

`train_lora` (T4 low-priority) -> `merge_lora` (CPU). No `prep` step (the dataset is already built),
no `evaluate` step (quality is out of scope).

The GPU is **low priority**, and that is often the only way in: dedicated GPU quota is frequently 0 on
modern families, while the low-priority pool is a separate allocation that does work. Price of
admission: preemption - acceptable for 10 minutes of training, impossible for an endpoint.

> Set `MAX_SAMPLES=40` in `.env` for a ~3-minute smoke run that proves the plumbing without renting
> the GPU for long. `0` uses all 500 samples of the train split, which is ~15 minutes.


In [ ]:
from azure.ai.ml import Input, load_job

# MAX_SAMPLES comes from .env (0 = the whole 500-sample train split).
pipeline = load_job(REPO / "finetune" / "pipelines" / "finetune_pipeline.yml")

# Attribute syntax, not pipeline.inputs["x"] = ... - this is not cosmetic.
# pipeline.inputs is an InputsAttrDict, and it overrides __setattr__ to unwrap the
# value into the existing PipelineInput (original._data = original._build_data(v)).
# It does NOT override __setitem__, so bracket assignment falls through to plain
# dict.__setitem__ and *replaces* the PipelineInput wrapper with a bare Input.
# Nothing complains until submission, where the SDK calls input._to_job_input() on
# every entry and a bare Input has no such method:
#     AttributeError: 'Input' object has no attribute '_to_job_input'
# (OutputsAttrDict does define __setitem__. Inputs simply never got it.)
pipeline.inputs.base_model = Input(
    type=AssetTypes.CUSTOM_MODEL, path=f"azureml:{base_model.name}:{base_model.version}"
)
pipeline.inputs.dataset = Input(type=AssetTypes.URI_FOLDER, path=f"azureml:{dataset.name}:{dataset.version}")
pipeline.inputs.max_samples = MAX_SAMPLES

job = ml_client.jobs.create_or_update(pipeline)
print(job.name)
print(job.studio_url)


In [ ]:
# Blocks until the run finishes and streams the logs of both steps.
# What to show meanwhile: the pipeline graph, then the loss curve in the
# Metrics tab of the train_lora step.
ml_client.jobs.stream(job.name)
print(ml_client.jobs.get(job.name).status)


---
## Brick 4: two artifacts from one run

We register both pipeline outputs as two distinct assets, with **cross tags**. The `base_model` tag is
not documentation: 10 MB of LoRA weights are a *delta*, and a delta without its starting point means
nothing. **That tag is the artifact's contract.**

Registration happens here rather than in a pipeline component: a component would have to
re-authenticate to the workspace, and would go through MLflow - which stamps the assets as
`mlflow_model` and breaks the `score.py` serving path.


In [ ]:
lineage = {
    "base_model": f"{base_model.name}:{base_model.version}",
    "base_repo": source["repo"],
    "base_revision": source["revision"],
    "dataset": f"{dataset.name}:{dataset.version}",
    "lora_r": "8",
    "lora_alpha": "16",
    "job": job.name,
}

adapter = ml_client.models.create_or_update(
    Model(
        name="maint-vlm-adapter",
        path=f"azureml://jobs/{job.name}/outputs/adapter",
        type=AssetTypes.CUSTOM_MODEL,
        description="LoRA adapter only. Unusable without its base model - see the base_model tag.",
        tags={**lineage, "artifact": "adapter"},
    )
)
merged = ml_client.models.create_or_update(
    Model(
        name="maint-vlm-merged",
        path=f"azureml://jobs/{job.name}/outputs/merged",
        type=AssetTypes.CUSTOM_MODEL,
        description="Base weights with the adapter merged in. Self-contained, servable as-is.",
        tags={**lineage, "artifact": "merged"},
    )
)
print(f"{adapter.name}:{adapter.version}")
print(f"{merged.name}:{merged.version}")


In [ ]:
# The single most legible visual in the whole demo: the two sizes, side by side.
import shutil

ARTIFACTS = REPO / "tmp-artifacts"
shutil.rmtree(ARTIFACTS, ignore_errors=True)


def folder_mb(path: Path) -> float:
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e6


for asset in (adapter, merged):
    ml_client.models.download(name=asset.name, version=asset.version, download_path=str(ARTIFACTS))

print(f"{'base model':24s} {source['size_mb']:8.1f} MB")
for asset in (adapter, merged):
    print(f"{asset.name:24s} {folder_mb(ARTIFACTS / asset.name):8.1f} MB   (v{asset.version})")


---
## Brick 5: $ serve the fine-tuned model in AML

One endpoint, one deployment, **CPU**. At 256M parameters a `Standard_DS3_v2` answers in a couple of
seconds, and it removes the 24/7 GPU cost question entirely.

> A managed online endpoint **never scales to zero**. Create it late, delete it early.
> The first deployment takes ~10 min (image build + pull).

`score.py` is explicit rather than MLflow no-code: the no-code path drags in its own serving stack,
and it has nothing useful to say about a VLM anyway.


In [ ]:
from azure.ai.ml.entities import (
    CodeConfiguration,
    ManagedOnlineDeployment,
    ManagedOnlineEndpoint,
    OnlineRequestSettings,
    ProbeSettings,
)

# ENDPOINT comes from .env. Endpoint names must be unique per region and
# subscription, so set ENDPOINT_NAME there if this one is taken.
ml_client.online_endpoints.begin_create_or_update(
    ManagedOnlineEndpoint(name=ENDPOINT, auth_mode="key", description="Fine-tuned maintenance VLM.")
).result()

# The probes are generous on purpose: loading transformers on CPU comfortably
# exceeds the default delays, and a perfectly healthy deployment then gets
# killed before it ever answers once.
ml_client.online_deployments.begin_create_or_update(
    ManagedOnlineDeployment(
        name="default",
        endpoint_name=ENDPOINT,
        model=f"azureml:{merged.name}:{merged.version}",
        environment=f"azureml:vlm-serve-env:{envs['vlm-serve-env']}",
        code_configuration=CodeConfiguration(
            code=str(REPO / "finetune" / "endpoints"), scoring_script="score.py"
        ),
        instance_type="Standard_DS3_v2",
        instance_count=1,
        request_settings=OnlineRequestSettings(request_timeout_ms=90000, max_concurrent_requests_per_instance=1),
        liveness_probe=ProbeSettings(initial_delay=600, period=30, timeout=10, failure_threshold=30),
        readiness_probe=ProbeSettings(initial_delay=600, period=30, timeout=10, failure_threshold=30),
    )
).result()

endpoint = ml_client.online_endpoints.get(ENDPOINT)
endpoint.traffic = {"default": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"{ENDPOINT} ready, traffic -> default 100%")


In [ ]:
# The inference: a chart in, a JSON work order out.
import base64
import json
import tempfile

IMAGE = REPO / "finetune" / "data" / "sample_chart.png"

request = Path(tempfile.gettempdir()) / "vlm_request.json"
request.write_text(json.dumps({"image_b64": base64.b64encode(IMAGE.read_bytes()).decode()}), encoding="utf-8")

response = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT, deployment_name="default", request_file=str(request)
)
print(json.dumps(json.loads(response), indent=2, ensure_ascii=False))


---
## Brick 6: detach the adapter

What the fine-tuning actually produced is not 500 MB of weights: it is **~10 MB** of delta
(2.44M trainable parameters out of 259M). Same image as above, same output, **outside Azure, on CPU**.

> These cells need `torch`, `transformers` and `peft` locally - that is what
> `requirements-local.txt` installs, in the first cell of this notebook.
> See [README-portability.md](../export/README-portability.md) for the other targets
> (vLLM, GGUF/Ollama, ONNX on an AI PC, another AML registry).


In [ ]:
import subprocess

adapter_dir = next((ARTIFACTS / adapter.name).rglob("adapter_config.json")).parent
print(f"adapter  : {adapter_dir}  ({folder_mb(adapter_dir):.1f} MB)")
print(f"contents : {sorted(p.name for p in adapter_dir.iterdir())}")

result = subprocess.run(
    [
        sys.executable,
        str(REPO / "finetune" / "export" / "run_local_adapter.py"),
        "--adapter",
        str(adapter_dir),
        "--image",
        str(IMAGE),
    ],
    capture_output=True,
    text=True,
)
print(result.stdout or result.stderr[-2000:])


### The counter-example, in 10 seconds

Same image, same prompt, but **without** the adapter: the base model chatters in prose instead of
returning JSON. That is exactly the difference the fine-tuning bought - and it fits in 10 MB.


In [ ]:
result = subprocess.run(
    [
        sys.executable,
        str(REPO / "finetune" / "export" / "run_local_adapter.py"),
        "--adapter",
        str(adapter_dir),
        "--image",
        str(IMAGE),
        "--no_adapter",
    ],
    capture_output=True,
    text=True,
)
print(result.stdout or result.stderr[-2000:])

### Measuring the difference, on charts the model has never seen

One anecdote is not a result. This scores both models on the **test split** - windows chronologically
later than everything in training - and draws the comparison.

**Be precise about the claim.** 500 samples on a 256M model do not produce a maintenance expert. What
a LoRA of this size teaches, and teaches remarkably well, is a **format and a vocabulary**: emit JSON,
use these five fields, pick from these enums. So the scoreboard leads with *valid JSON* and
*schema OK* - binary, undeniable, and exactly what you buy for 10 MB. Field agreement comes second,
and the confusion matrix is there to show honestly where the model still guesses.

The scoring runs **locally, on CPU, from the adapter** - no endpoint, no GPU, no Azure call. Which is
itself the argument of brick 6.


In [ ]:
from IPython.display import Image as Show

# DATASET_LOCAL is the folder built in brick 2. We score the TEST split: those
# windows are later in time than every training window, so the model has never
# seen them - and the scoring runs locally, on CPU, from the adapter. No endpoint,
# no GPU, no Azure call, which is itself the argument of brick 6.
EVAL_OUT = REPO / "tmp-eval"
N_SCORED = 24  # 2 generations each, CPU: a few minutes

result = subprocess.run(
    [
        sys.executable, str(REPO / "finetune" / "export" / "eval_visual.py"),
        "--adapter", str(adapter_dir),
        "--dataset", str(DATASET_LOCAL),
        "--n", str(N_SCORED),
        "--out", str(EVAL_OUT),
    ],
    capture_output=True,
    text=True,
)
print(result.stdout[-3000:] or result.stderr[-3000:])


In [ ]:
# The slide: base vs fine-tuned, and where the fine-tuned model still guesses.
Show(filename=str(EVAL_OUT / "eval_scoreboard.png"))


In [ ]:
# The same charts, prose on the left, work orders on the right.
Show(filename=str(EVAL_OUT / "eval_examples.png"))


---
## teardown - run this as soon as the demo is over

The online endpoint is the **only** thing here that bills continuously. The GPU cluster drops back to
0 nodes on its own after 5 minutes idle, but we check it.

We delete the **endpoint**, which cascades to its deployments. Deleting the deployment first is the
tempting order and the wrong one: a deployment holding traffic refuses to be deleted, so the teardown
fails at the exact moment you want it to be boring. The cell is also re-runnable - it checks what
exists before deleting - and it redefines `ENDPOINT` locally, so it can be run alone on a fresh
kernel. That is the cell you run at 11pm when you remember you left something on.

Set `CONFIRM = True` to execute.


In [ ]:
CONFIRM = True

# Self-contained on purpose: this is the cell you run alone, on a fresh kernel, at
# 11pm, when you remember you left something on. It re-reads .env and rebuilds its
# own client rather than depending on a single cell above it.
import os
from pathlib import Path

from azure.ai.ml import MLClient
from azure.core.exceptions import ResourceNotFoundError
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

_repo = Path.cwd()
while not (_repo / ".env.example").is_file() and _repo != _repo.parent:
    _repo = _repo.parent
load_dotenv(_repo / ".env", override=False)

endpoint_name = os.environ.get("ENDPOINT_NAME", "maint-vlm-ep")
client = MLClient(
    DefaultAzureCredential(),
    os.environ["AZURE_SUBSCRIPTION_ID"],
    os.environ["AZURE_RESOURCE_GROUP"],
    os.environ["AZUREML_WORKSPACE_NAME"],
)

try:
    cluster = client.compute.get("gpu-cluster-spot")
    nodes = client.compute.list_nodes("gpu-cluster-spot")
    print(
        f"gpu-cluster-spot : min={cluster.min_instances} idle={cluster.idle_time_before_scale_down}s "
        f"-> {len(list(nodes))} node(s) allocated"
    )
except ResourceNotFoundError:
    print("gpu-cluster-spot : does not exist - nothing can be billing")

live = [e.name for e in client.online_endpoints.list()]
print(f"endpoints        : {live}")

# Delete the ENDPOINT, not the deployment. A deployment holding traffic refuses to
# be deleted ("Can't delete deployment with non-zero traffic weight", buried three
# levels down in the error), and deleting the endpoint cascades to every deployment
# under it anyway. Only our endpoint is touched; anything else in the workspace is
# left alone.
if not CONFIRM:
    print("CONFIRM=False: nothing deleted. The endpoint bills for as long as it exists.")
elif endpoint_name not in live:
    print(f"{endpoint_name} already gone - nothing to bill, nothing to do.")
else:
    client.online_endpoints.begin_delete(name=endpoint_name).result()
    print(f"{endpoint_name} deleted (with its deployments)")
